# Book RAG Project

Your task is to build a **retrieval-augmented generation (RAG)** system that answers questions about **one book**, and shows the passages of the book that each answer is based on. Every group works on a different book from Project Gutenberg; the table in the README shows which book is yours.

![The project in two parts. Part 1, once per book: 1. get your book, 2. clean the text, 3. split it into chunks, 4. embed the chunks and store them in a Chroma vector store. Part 2, for each question: 5. retrieve the passages from the vector store, 6. generate the answer, and return the answer with its passages](assets/book_rag_project_workflow.png)

# The Business Problem

A public library wants to offer a **reading companion** on its website: readers ask a question about a book in their own words, for example "Why does the Tin Woodman want a heart?", and receive a short answer. Readers must be able to check every answer, so the companion also shows **the passages of the book that the answer is based on**. When the book does not contain the answer, the companion has to say so instead of making one up.

A large language model alone already knows a lot about famous books, but it does not cite passages, and it can confuse details or invent them, especially for less well-known books. Your RAG system retrieves the passages from the book first and lets the model answer **only from these passages**, so that every answer comes with the passages a reader can check.

Before the library builds the website, its staff want to **try the companion themselves**. Your last task is therefore a **command-line prototype**: a small script that librarians run in the terminal to ask questions about your book and read the answers with their passages.

# Data Description

The data ships with this repository in the folder `data/`: **eight books** from [Project Gutenberg](https://www.gutenberg.org/), one plain text file per book (UTF-8). Your group works with **one** of them.

| File | Title | Author | First published | Kind | Words |
| :-- | :-- | :-- | :-- | :-- | --: |
| `adventures_of_pinocchio_collodi.txt` | The Adventures of Pinocchio | Carlo Collodi | 1883 | children's novel, translated from Italian | 39,900 |
| `adventures_of_sherlock_holmes_doyle.txt` | The Adventures of Sherlock Holmes | Arthur Conan Doyle | 1892 | twelve detective stories | 104,500 |
| `antarctic_penguins_levick.txt` | Antarctic Penguins: A Study of Their Social Habits | G. Murray Levick | 1914 | natural history | 30,600 |
| `climate_in_relation_to_man_ward.txt` | Climate, Considered Especially in Relation to Man | Robert DeCourcy Ward | 1918 | climatology textbook (second edition) | 88,300 |
| `frankenstein_shelley.txt` | Frankenstein; or, The Modern Prometheus | Mary Wollstonecraft Shelley | 1818 | gothic novel | 75,000 |
| `the_time_machine_wells.txt` | The Time Machine | H. G. Wells | 1895 | science fiction novella | 32,500 |
| `tom_browns_school_days_hughes.txt` | Tom Brown's School Days | Thomas Hughes | 1857 | boarding school novel | 105,400 |
| `wonderful_wizard_of_oz_baum.txt` | The Wonderful Wizard of Oz | L. Frank Baum | 1900 | children's fantasy novel | 39,600 |

The word counts leave out the licence text. Every file is exactly as Project Gutenberg distributes it:

- It **starts with a header** (title, author, release date) and the line `*** START OF THE PROJECT GUTENBERG EBOOK ... ***`, and it **ends with** the line `*** END OF THE PROJECT GUTENBERG EBOOK ... ***`, followed by the Project Gutenberg licence. Only the text between these two lines belongs to the book.
- The lines are **wrapped after about 70 characters**, so a single sentence is spread over several lines. A blank line separates two paragraphs.
- Some books contain notes of the transcribers, a table of contents, and placeholders such as `[Illustration]`.

The books are in the public domain in the United States; the Project Gutenberg licence at the end of each file explains how they may be used.

## Task

1. **Get your book.** Set `BOOK` to the file name of your book and read it with `Path.read_text`. Print the first and last 1,500 characters: what comes before and after the book itself?
2. **Clean the text.** Keep only the text between the `*** START OF ...` and `*** END OF ...` lines, and join the wrapped lines of each paragraph with `re.sub`. Store the result in a [`Document`](https://reference.langchain.com/python/langchain-core/documents/base/Document) with the title and author in its `metadata`.
3. **Split into chunks.** Use [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) with a `chunk_size` and `chunk_overlap` of your choice, and `add_start_index=True`, so that every chunk stores its position in the book. How many chunks do you get?
4. **Embed and store.** Embed the chunks with [`HuggingFaceEmbeddings`](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings) (`sentence-transformers/all-MiniLM-L6-v2`) and store them in a [`Chroma`](https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma) vector store with `persist_directory="chroma_db"`.

> [!IMPORTANT]
> Pass your own `ids` to `add_documents`, for example `f"{BOOK}-{i}"`. Without them, every run of the cell adds all chunks again.

5. **Retrieve.** Create a retriever with `as_retriever(search_kwargs={"k": 4})` and `invoke` it with a few questions about your book. Print the chunks with their `start_index`: do they contain the answer?
6. **Generate.** Build an LCEL chain `prompt | llm | parser` from a [`ChatPromptTemplate`](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate) that tells the model to answer **only from the passages**, [`ChatGroq`](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) with `openai/gpt-oss-20b` and `temperature=0`, and a `StrOutputParser`. Write `ask(question)`, which returns the answer and the `start_index` of each passage. Try your questions from step 5, and one whose answer is not in the book.
7. **Build the command-line prototype.** Copy your code into `rag_cli.py` and complete the `while` loop described in its comments, so that it reads questions with `input()` until the user types `quit`, and prints each answer and its passage in colour. Run it with `uv run python rag_cli.py`.

> [!NOTE]
> The free Groq plan limits requests per minute. If you get a rate limit error, wait a minute and try again.

In [ ]:
# Import the libraries you need for this project
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # reads GROQ_API_KEY from the .env file

In [ ]:
# Your code here